# 03 - Silver: reviews_limpo
Limpeza e padronização das avaliações brutas.
- Entrada: `bronze.reviews`
- Saída: `silver.reviews_limpo`

In [0]:
from pyspark.sql import functions as F

CATALOG = "voc_project"
BRONZE_SCHEMA = f"{CATALOG}.bronze"
SILVER_SCHEMA = f"{CATALOG}.silver"

AMOSTRA = 3000   # nº de reviews a processar (controla custo/tempo das próximas etapas)

In [0]:
df = spark.table(f"{BRONZE_SCHEMA}.reviews")

# Renomear colunas (bronze gravou com underscore)
renomear = {
    "_c0": "review_id", "Clothing_ID": "clothing_id", "Age": "age", "Title": "title",
    "Review_Text": "review_text", "Rating": "rating", "Recommended_IND": "recommended",
    "Positive_Feedback_Count": "positive_feedback_count", "Division_Name": "division_name",
    "Department_Name": "department_name", "Class_Name": "class_name",
}
for antigo, novo in renomear.items():
    if antigo in df.columns:
        df = df.withColumnRenamed(antigo, novo)

# Tipagem
df = (df
    .withColumn("age", F.expr("try_cast(age as int)"))
    .withColumn("rating", F.expr("try_cast(rating as int)"))
    .withColumn("recommended", F.expr("try_cast(recommended as int)"))
    .withColumn("positive_feedback_count", F.expr("try_cast(positive_feedback_count as int)"))
)

# Remover reviews sem texto (não é possível embedar vazio)
antes = df.count()
df = df.filter(F.col("review_text").isNotNull() & (F.trim(F.col("review_text")) != ""))
print(f"Removidas {antes - df.count()} reviews sem texto")

# Campo único de texto (título + corpo)
df = df.withColumn(
    "texto_completo",
    F.trim(F.concat_ws(". ", F.coalesce(F.col("title"), F.lit("")), F.col("review_text")))
)

# Amostra
df_amostra = df.limit(AMOSTRA)
print(f"Amostra: {df_amostra.count()} reviews")

In [0]:
df_amostra.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{SILVER_SCHEMA}.reviews_limpo")

print(f"✅ silver.reviews_limpo: {df_amostra.count()} reviews")

In [0]:
print("Colunas:", df_amostra.columns)
display(df_amostra.select("review_id", "texto_completo", "rating", "department_name").limit(5))

# 04 - Silver: reviews_embeddings
Gera embeddings (vetor de significado) para cada review via Foundation Model API.
- Entrada: `silver.reviews_limpo`
- Saída: `silver.reviews_embeddings`

In [0]:
from pyspark.sql import functions as F

CATALOG = "voc_project"
SILVER_SCHEMA = f"{CATALOG}.silver"

EMBEDDING_ENDPOINT = "databricks-gte-large-en"

## Gerar embeddings (em lote via ai_query)
Cada review vira um vetor de 1024 dimensões que captura seu significado.

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {SILVER_SCHEMA}.reviews_embeddings AS
    SELECT
        *,
        ai_query('{EMBEDDING_ENDPOINT}', texto_completo) AS embedding
    FROM {SILVER_SCHEMA}.reviews_limpo
""")

print("✅ Embeddings gerados")

In [0]:
# from pyspark.sql.functions import expr

# df_embeddings = (
#     df_amostra
#     .withColumn(
#         "embedding",
#         expr(f"ai_query('{EMBEDDING_ENDPOINT}', texto_completo)")
#     )
# )

# display(df_embeddings)

## Validação

In [0]:
df_emb = spark.table(f"{SILVER_SCHEMA}.reviews_embeddings")
#df_emb = df_embeddings

exemplo = df_emb.select("embedding").first()["embedding"]
print("Dimensões do embedding:", len(exemplo))
print("Reviews com embedding:", df_emb.filter(F.col("embedding").isNotNull()).count())
print("Total de reviews:", df_emb.count())

display(df_emb.select("texto_completo", "embedding").limit(3))

# 05 - Silver: reviews_enriched
Classifica cada review em **ASSUNTO + SENTIMENTO** numa única chamada de LLM (JSON).
- Entrada: `silver.reviews_embeddings`
- Saída: `silver.reviews_enriched`

Nota: produto/departamento não é classificado por LLM — já existe nas colunas originais do dataset.

In [0]:
from pyspark.sql import functions as F

CATALOG = "voc_project"
SILVER_SCHEMA = f"{CATALOG}.silver"

LLM_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

In [0]:
# spark.sql(f"""
#     CREATE OR REPLACE TABLE {SILVER_SCHEMA}.reviews_classified_raw AS
#     SELECT
#         *,
#         ai_query(
#             '{LLM_ENDPOINT}',
#             CONCAT(
#                 'Analise esta avaliação de roupa e responda APENAS com um JSON válido, sem texto extra, no formato: ',
#                 '{{"assunto": "...", "sentimento": "..."}}. ',
#                 'Regras: ',
#                 '"assunto" = o tema principal do comentário, geralmente um problema ou aspecto avaliado ',
#                 '(ex: caimento, tamanho, qualidade do tecido, cor, conforto, defeito, preço). Use 1 a 3 palavras. ',
#                 '"sentimento" = positivo, negativo ou neutro. ',
#                 'Responda em português. Avaliação: ',
#                 texto_completo
#             )
#         ) AS classificacao_json
#     FROM {SILVER_SCHEMA}.reviews_embeddings
# """)

# print("✅ Classificação bruta (JSON) gerada")
# display(spark.table(f"{SILVER_SCHEMA}.reviews_classified_raw").select("texto_completo", "classificacao_json").limit(5))

In [0]:
spark.sql(f"""
    CREATE OR REPLACE TABLE {SILVER_SCHEMA}.reviews_classified_raw AS
    SELECT
        *,
        ai_query(
            '{LLM_ENDPOINT}',
            CONCAT(
                'Você classifica avaliações de roupas. Responda APENAS com um JSON válido, ',
                'sem texto extra, no formato: {{"assunto": "...", "sentimento": "..."}}.\\n\\n',
                'O campo "assunto" DEVE ser EXATAMENTE um destes valores (escolha o mais relevante):\\n',
                '- caimento\\n',
                '- tamanho\\n',
                '- conforto\\n',
                '- qualidade\\n',
                '- tecido\\n',
                '- cor\\n',
                '- estilo\\n',
                '- comprimento\\n',
                '- preco\\n',
                '- transparencia\\n',
                '- encolhimento\\n',
                '- outro\\n\\n',
                'Regras: escolha UM único assunto da lista acima, exatamente como escrito ',
                '(minúsculas, sem acento). Se não encaixar em nenhum, use "outro". ',
                'NÃO invente assuntos fora da lista. NÃO use o tipo de peça como assunto.\\n\\n',
                'O campo "sentimento" deve ser: positivo, negativo ou neutro.\\n\\n',
                'Avaliação: ', texto_completo
            )
        ) AS classificacao_json
    FROM {SILVER_SCHEMA}.reviews_embeddings
""")

print("✅ Classificação bruta (JSON) gerada")
display(spark.table(f"{SILVER_SCHEMA}.reviews_classified_raw").select("texto_completo", "classificacao_json").limit(5))

## Extrair campos do JSON e normalizar sentimento

In [0]:
df = spark.table(f"{SILVER_SCHEMA}.reviews_classified_raw")
#f = df_classified

df_extraido = (df
    .withColumn("assunto", F.lower(F.trim(F.get_json_object(F.col("classificacao_json"), "$.assunto"))))
    .withColumn("sentimento", F.lower(F.trim(F.get_json_object(F.col("classificacao_json"), "$.sentimento"))))
    # normaliza sentimento pras 3 categorias
    .withColumn("sentimento",
        F.when(F.col("sentimento").contains("positiv"), "positivo")
         .when(F.col("sentimento").contains("negativ"), "negativo")
         .when(F.col("sentimento").contains("neutr"), "neutro")
         .otherwise("indefinido"))
)

df_extraido.write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{SILVER_SCHEMA}.reviews_enriched")

print("✅ silver.reviews_enriched gravada")

## Validação
Checagem pra confirmar que o LLM respeitou a lista e tambem da distribuicao de assuntos e sentimentos.

In [0]:
# confere se algum assunto ficou fora da lista permitida
assuntos_validos = ["caimento", "tamanho", "conforto", "qualidade", "tecido",
                    "cor", "estilo", "comprimento", "preco", "transparencia",
                    "encolhimento", "outro"]

df_check = spark.table(f"{SILVER_SCHEMA}.reviews_enriched")
fora_da_lista = df_check.filter(~F.col("assunto").isin(assuntos_validos))
print("Assuntos fora da lista (deveria ser 0 ou poucos):")
fora_da_lista.groupBy("assunto").count().show(truncate=False)

In [0]:
df_final = spark.table(f"{SILVER_SCHEMA}.reviews_enriched")
#df_final = df_extraido

# Quantos ficaram indefinidos (JSON mal formado)
indefinidos = df_final.filter(F.col("sentimento") == "indefinido").count()
sem_assunto = df_final.filter(F.col("assunto").isNull()).count()
print(f"Sentimento indefinido: {indefinidos}")
print(f"Sem assunto: {sem_assunto}")

print("\nDistribuição de sentimento:")
df_final.groupBy("sentimento").count().orderBy(F.desc("count")).show()

print("Top assuntos:")
df_final.groupBy("assunto").count().orderBy(F.desc("count")).show(15, truncate=False)

## Validação cruzada (sanity check)
Sentimento negativo deveria ter rating médio baixo — se crescer de negativo→positivo, o LLM está coerente.

In [0]:
print("Rating médio por sentimento (deve crescer negativo→positivo):")
df_final.groupBy("sentimento") \
    .agg(F.round(F.avg("rating"), 2).alias("rating_medio"),
         F.count("*").alias("qtd")) \
    .orderBy("rating_medio") \
    .show()

f

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {SILVER_SCHEMA}.reviews_limpo")
#spark.sql(f"DROP TABLE IF EXISTS {SILVER_SCHEMA}.reviews_embeddings")
spark.sql(f"DROP TABLE IF EXISTS {SILVER_SCHEMA}.reviews_classified_raw")

print("✅ Tabelas apagadas")